In [17]:
import os
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [15]:
load_dotenv()

OLLAMA_MODEL    = "llama3.2:latest"         
OLLAMA_EMBED    = "nomic-embed-text" 
OLLAMA_BASE_URL = "http://localhost:11434"

In [ ]:
RAG_DOCUMENTS = [
    """Pasta al Pomodoro : Faites cuire des pâtes al dente. Faites revenir de l'ail dans l'huile d'olive,
    ajoutez des tomates concassées, sel, basilic frais. Mélangez avec les pâtes et servez avec du parmesan.""",

    """Omelette aux légumes : Battez des œufs avec sel et poivre. Faites revenir les légumes disponibles
    (poivron, oignon, courgette) dans du beurre. Versez les œufs, pliez quand c'est pris.""",

    """Soupe minestrone : Faites revenir oignon, carotte, céleri. Ajoutez bouillon, tomates, courgettes,
    haricots, pâtes courtes. Cuisez 20 minutes. Servez avec du parmesan râpé.""",

    """Pizza maison rapide : Étalez de la pâte à pizza, nappez de sauce tomate, ajoutez mozzarella et
    vos garnitures. Cuisez à 220°C pendant 12-15 minutes jusqu'à ce que le fromage soit doré.""",

    """Risotto aux champignons : Faites revenir oignon, ajoutez riz arborio, versez du vin blanc.
    Ajoutez le bouillon chaud louche par louche en remuant. Incorporez champignons sautés, parmesan, beurre.""",

    """Salade caprese : Alternez tranches de tomates fraîches et mozzarella. Assaisonnez d'huile d'olive
    extra-vierge, sel, poivre, et basilic frais. Simple et délicieux.""",

    """Curry de légumes : Faites revenir oignon et ail. Ajoutez pâte de curry, lait de coco, légumes de saison
    (courgette, poivron, patate douce). Cuisez 20 minutes. Servez avec du riz basmati.""",

    """Quiche lorraine : Foncez un moule de pâte brisée. Mélangez œufs, crème, lardons, gruyère râpé,
    sel, poivre, muscade. Versez, cuisez 35 min à 180°C.""",
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks   = splitter.create_documents(RAG_DOCUMENTS)

embeddings  = OllamaEmbeddings(model=OLLAMA_EMBED, base_url=OLLAMA_BASE_URL)
vectorstore = Chroma.from_documents(chunks, embeddings)
print(f"  {len(chunks)} chunks indexés")

  8 chunks indexés


In [41]:
@tool
def search_recipe_web(query=None) -> str:
    """Recherche des recettes et informations culinaires sur le web via Tavily.
    Utilise cet outil quand tu as besoin d'informations précises sur une recette ou technique."""
    if isinstance(query, list):
        query = ", ".join(str(q) for q in query)
    elif not isinstance(query, str):
        query = str(query)
    try:
        from tavily import TavilyClient
        client   = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        response = client.search(query=query, max_results=3)
        results  = []
        for r in response["results"]:
            results.append(f"{r['title']}\n{r['content'][:500]}\nSource: {r['url']}")
        return "\n\n---\n\n".join(results) if results else "Aucun résultat trouvé."
    except Exception as e:
        return f"Erreur Tavily: {e}. Utilise tes connaissances internes."


@tool
def search_recipe_rag(query=None) -> str:
    """Recherche dans la base de connaissances locale (RAG) des recettes et techniques culinaires.
    Utilise cet outil en priorité avant la recherche web."""
    if isinstance(query, list):
        query = ", ".join(str(q) for q in query)
    elif not isinstance(query, str):
        query = str(query)
    docs    = vectorstore.similarity_search(query, k=3)
    results = [f"Résultat {i+1}:\n{doc.page_content}" for i, doc in enumerate(docs)]
    return "\n\n".join(results) if results else "Aucune recette trouvée dans la base locale."


@tool
def save_preference(preference=None, **kwargs) -> str:
    """Sauvegarde une préférence ou information importante de l'utilisateur en mémoire.
    Exemples: allergies, régime alimentaire, plats favoris, restrictions.
    Appelle cet outil UNE FOIS avec une string résumant toutes les préférences."""
    if isinstance(preference, list):
        preference = ", ".join(str(p) for p in preference)
    elif isinstance(preference, dict):
        preference = ", ".join(f"{k}: {v}" for k, v in preference.items())
    elif not preference and kwargs:
        preference = ", ".join(f"{k}: {v}" for k, v in kwargs.items())
    return f"Préférence mémorisée : '{preference}'"


tools     = [search_recipe_web, search_recipe_rag, save_preference]
tools_map = {t.name: t for t in tools}

print("Outils définis :", list(tools_map.keys()))

Outils définis : ['search_recipe_web', 'search_recipe_rag', 'save_preference']


In [27]:
SYSTEM_PROMPT = """Tu es un chef cuisinier personnel expert et chaleureux.

Tes responsabilités :
1. **Ingrédients** : Analyser les ingrédients disponibles et proposer des recettes créatives et réalisables.
2. **Mémoire** : Retenir et respecter les préférences, allergies et restrictions alimentaires de l'utilisateur.
   Utilise l'outil `save_preference` pour mémoriser ces informations importantes.
3. **Recettes** : Utilise d'abord `search_recipe_rag` pour chercher dans ta base locale,
   puis `search_recipe_web` si tu as besoin d'informations supplémentaires.
4. **Pédagogie** : Explique les étapes de préparation clairement, avec des conseils de chef.

Ton style :
- Propose toujours 2-3 options de plats quand c'est possible.
- Indique le temps de préparation et la difficulté.
- Donne des astuces pour sublimer le plat.
- Sois enthousiaste et encourageant !
"""

model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.7,
).bind_tools(tools)

print(f" Modèle Ollama prêt : {OLLAMA_MODEL}")

 Modèle Ollama prêt : llama3.2:latest


In [38]:
def run_agent(messages: list, user_preferences: list, user_input: str):
    """
    Boucle agent manuelle :
      1. Ajoute le message utilisateur
      2. Appelle le modèle
      3. Si tool_calls → exécute les outils → rappelle le modèle
      4. Répète jusqu'à réponse finale
    """
    system_content = SYSTEM_PROMPT
    if user_preferences:
        prefs_text     = "\n".join(f"- {p}" for p in user_preferences)
        system_content += f"\n\nPréférences mémorisées de l'utilisateur:\n{prefs_text}"

    system_msg = SystemMessage(content=system_content)

    messages = messages + [HumanMessage(content=user_input)]

    while True:
        response = model.invoke([system_msg] + messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tc in response.tool_calls:
            tool_name   = tc["name"]
            tool_args   = tc["args"]
            tool_id     = tc["id"]

            print(f" Outil appelé : {tool_name}({tool_args})")

            tool_fn     = tools_map[tool_name]
            tool_result = tool_fn.invoke(tool_args)

            if tool_name == "save_preference" and "Préférence mémorisée" in str(tool_result):
                if "preference" in tool_args:
                    pref = tool_args["preference"]
                else:
                    pref = ", ".join(f"{k}: {v}" for k, v in tool_args.items())
                if pref and pref not in user_preferences:
                    user_preferences.append(pref)

            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))

    return messages, user_preferences



In [39]:
messages     = []
preferences  = []

user_msg_1 = "Bonjour ! Je suis végétarien et je suis allergique aux noix."
print(f" Utilisateur : {user_msg_1}\n")

messages, preferences = run_agent(messages, preferences, user_msg_1)

print(f"\n Chef : {messages[-1].content}")
print(f"\n Préférences mémorisées : {preferences}")

 Utilisateur : Bonjour ! Je suis végétarien et je suis allergique aux noix.

 Outil appelé : save_preference({'kwargs': {}, 'preference': 'végétarien et allergique aux noix'})

 Chef : Bonjour ! Je suis ravi de vous aider à trouver des recettes végétariennes délicieuses sans noix. Étant donné votre allergie, je vais être très prudent.

Voici trois options de plats que je vous propose :

**Option 1 : Soupe d' légumes et quinoa**
Temps de préparation : 30 minutes
Difficulté : Facile

Cette soupe est faite avec une variété d' légumes tels que des carottes, des pommes de terre et des pois. Le quinoa ajoute une texture délicieuse et une bonne source de protéines.

**Option 2 : Tacos aux légumes**
Temps de préparation : 45 minutes
Difficulté : Moyenne

Ces tacos sont remplis avec une combinaison d' légumes tels que des tomates, des concombres et des poivrons. Vous pouvez ajouter un peu de sauce végétale pour donner plus de saveur.

**Option 3 : Salade de riz et légumes**
Temps de préparation

In [42]:
user_msg_2 = """
Voici ce que j'ai dans mon réfrigérateur :
- tomates
- fromage (mozzarella)
- pâtes
- basilic frais
- ail
- huile d'olive

Qu'est-ce que tu me proposes ?
"""
print(f"Utilisateur : {user_msg_2.strip()}\n")

messages, preferences = run_agent(messages, preferences, user_msg_2)

print(f"\n Chef : {messages[-1].content}")

Utilisateur : Voici ce que j'ai dans mon réfrigérateur :
- tomates
- fromage (mozzarella)
- pâtes
- basilic frais
- ail
- huile d'olive

Qu'est-ce que tu me proposes ?

 Outil appelé : search_recipe_rag({'query': ['pátes avec tomates et fromage', 'salade de tomate et basilic']})

 Chef : Vous avez un excellent choix de ingrédients ! Voici mes recommandations :

**Option 1 : Salade caprese**
Temps de préparation : 15 minutes
Difficulté : Facile

Cette salade est simple mais délicieuse. Les tomates fraîches, le mozzarella et le basilic frais vont vous donner une saveur méditerranéenne.

**Option 2 : Pasta al Pomodoro**
Temps de préparation : 30 minutes
Difficulté : Moyenne

Cette recette est une classique italienne. Les tomates concassées ajoutent une saveur sourre qui va bien avec le fromage et l'ail.

Je vous propose ces deux options, car elles sont faciles à préparer et vont vous donner un bon goût. Quelle choisissez-vous ?

Astuces :
- Utilisez toujours des herbes fraîches pour donne

In [44]:
user_msg_3 = "Comment je fais pour que les pâtes soient parfaitement al dente ?"
print(f"Utilisateur : {user_msg_3}\n")

messages, preferences = run_agent(messages, preferences, user_msg_3)

print(f"\n Chef : {messages[-1].content}")

Utilisateur : Comment je fais pour que les pâtes soient parfaitement al dente ?

 Outil appelé : search_recipe_rag({'query': 'al dente pâtes'})


KeyboardInterrupt: 

In [45]:
print("=" * 50)
print(" Session terminée")
print(f" Messages dans l'historique : {len(messages)}")
print(f" Préférences mémorisées    : {preferences}")

 Session terminée
 Messages dans l'historique : 8
 Préférences mémorisées    : ['végétarien et allergique aux noix']
